In [ ]:
from datasets import load_dataset

ds = load_dataset("mteb/RuReviewsClassification")

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 15000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2048
    })
})

In [13]:
X_train, y_train = ds["train"]["text"], ds["train"]["label"]
X_test, y_test = ds["test"]["text"], ds["test"]["label"]

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline, make_union

features = make_union(
    TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=3,
        sublinear_tf=True,
        max_features=300_000,
    ),
)
clf = make_pipeline(features, LogisticRegression(C=4, max_iter=2000, class_weight="balanced"))

clf.fit(X_train, y_train)
print(classification_report(y_test, clf.predict(X_test), digits=3))

              precision    recall  f1-score   support

           0      0.748     0.705     0.726       682
           1      0.639     0.669     0.654       683
           2      0.867     0.876     0.871       683

    accuracy                          0.750      2048
   macro avg      0.751     0.750     0.750      2048
weighted avg      0.751     0.750     0.750      2048



In [30]:
import torch
from sklearn.metrics import classification_report
from transformers import pipeline

clf_bert = pipeline(
    "text-classification",
    model="seara/rubert-tiny2-russian-sentiment",
    device=0 if torch.cuda.is_available() else -1,
)

label2id = {"negative": 0, "neutral": 1, "positive": 2}

preds = clf_bert(list(X_test), batch_size=64, truncation=True, max_length=512)
y_pred = [label2id[p["label"]] for p in preds]

print(classification_report(y_test, y_pred, digits=3))

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 5474.97it/s]


              precision    recall  f1-score   support

           0      0.805     0.730     0.766       682
           1      0.681     0.764     0.720       683
           2      0.920     0.892     0.906       683

    accuracy                          0.795      2048
   macro avg      0.802     0.795     0.797      2048
weighted avg      0.802     0.795     0.797      2048



In [31]:
bert = pipeline(model="seara/rubert-tiny2-russian-sentiment", top_k=None)

message = "ресторан плохой"

logreg_pred = clf.predict_proba([message])
bert_pred = bert(message, top_K=None)

print("Baseline предсказание:")
print(logreg_pred)
print("RuBERT предсказание:")
print(bert_pred)

Loading weights: 100%|██████████| 57/57 [00:00<00:00, 8942.74it/s]


Baseline предсказание:
[[0.49887364 0.45250845 0.04861791]]
RuBERT предсказание:
[[{'label': 'negative', 'score': 0.5948731303215027}, {'label': 'neutral', 'score': 0.37992405891418457}, {'label': 'positive', 'score': 0.025202849879860878}]]


baseline модель показала неплохие метрики, но во первых есть ограничение словарного запаса, а также менее отчетливая уверенность по сравнению с bert. Нас интересует конкретно различие между положительным и отрицательным скором, поэтому я считаю высокая уверенность в конкретном классе благодаря комплексности модели преобладает над метриками в вакууме